[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openlayer-ai/openlayer-python/blob/main/examples/tracing/azure-speech/azure_speech_tracing.ipynb)


# <a id="top">Azure AI Speech tracing quickstart</a>

This notebook illustrates how to monitor Azure AI Speech (speech-to-text, speech translation and text-to-speech) with Openlayer.

The Speech SDK and the Openlayer tracer both run in your backend: the Azure key stays in your server-side configuration and is never sent to Openlayer. Only non-secret settings (region, language, voice, custom endpoint ID, output format), the recognized text or synthesized-audio stats, latency and failure details are traced. Do not ship either the Azure key or the Openlayer API key in browser or mobile code.

In [ ]:
!pip install openlayer azure-cognitiveservices-speech

## 1. Set the environment variables

In [ ]:
import os

# Azure AI Speech env variables (load these from your secret store in production)
os.environ["AZURE_SPEECH_KEY"] = "YOUR_AZURE_SPEECH_KEY_HERE"
os.environ["AZURE_SPEECH_REGION"] = "YOUR_AZURE_SPEECH_REGION_HERE"

# Openlayer env variables
os.environ["OPENLAYER_API_KEY"] = "YOUR_OPENLAYER_API_KEY_HERE"
os.environ["OPENLAYER_INFERENCE_PIPELINE_ID"] = "YOUR_OPENLAYER_INFERENCE_PIPELINE_ID_HERE"

## 2. Initialize Openlayer and create the Speech clients

`init()` auto-instruments every `SpeechRecognizer`, `TranslationRecognizer` and `SpeechSynthesizer` created afterwards. You can also trace a single client explicitly with `trace_azure_speech(client)`.

Audio is **not** sent to Openlayer by default. Set `attachment_upload_enabled=True` only if your security and privacy requirements allow storing audio in Openlayer; synthesized audio and any input audio you pass as `openlayer_audio` are then uploaded alongside the trace.

In [ ]:
import azure.cognitiveservices.speech as speechsdk

from openlayer.lib import init

init(
    attachment_upload_enabled=False,  # set to True to upload audio to Openlayer
)

speech_config = speechsdk.SpeechConfig(
    subscription=os.environ["AZURE_SPEECH_KEY"],
    region=os.environ["AZURE_SPEECH_REGION"],
)
speech_config.speech_recognition_language = "en-US"
speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"

## 3. Use your traced clients normally

### Text-to-speech

In [ ]:
synthesizer = speechsdk.SpeechSynthesizer(  # auto-traced by Openlayer
    speech_config=speech_config,
    audio_config=speechsdk.audio.AudioOutputConfig(filename="greeting.wav"),
)

result = synthesizer.speak_text("Hello! Thanks for calling. How can I help you today?")
result.reason

### Speech-to-text

The Speech SDK does not expose the audio behind an `AudioConfig`, so pass the same file as `openlayer_audio` if you want it attached to the trace (only used when attachment uploads are enabled).

In [ ]:
recognizer = speechsdk.SpeechRecognizer(  # auto-traced by Openlayer
    speech_config=speech_config,
    audio_config=speechsdk.audio.AudioConfig(filename="greeting.wav"),
)

result = recognizer.recognize_once(openlayer_audio="greeting.wav")
result.reason, result.text

### Grouping calls into one trace

Wrap a conversational turn in `@trace` to see recognition and synthesis as steps of the same trace.

In [ ]:
from openlayer.lib import trace


@trace()
def voice_turn(audio_path: str) -> str:
    recognizer = speechsdk.SpeechRecognizer(
        speech_config=speech_config,
        audio_config=speechsdk.audio.AudioConfig(filename=audio_path),
    )
    heard = recognizer.recognize_once(openlayer_audio=audio_path).text

    reply = f"You said: {heard}"
    speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=None).speak_text(reply)
    return reply


voice_turn("greeting.wav")

That's it! The Speech calls are published to Openlayer and you can start creating tests around them.